In [1]:
from google.cloud import bigquery
import os
import plotly.express as px
import pandas as pd
from pyproj import Transformer

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../data/auth/sante-et-territoires-216daad01024.json" 
client = bigquery.Client() 

query = """
SELECT *
FROM `sante-et-territoires.finess.equipements_sociaux_etabl`
"""

df = client.query(query).to_dataframe()



In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109365 entries, 0 to 109364
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   nofinesset        109365 non-null  object
 1   de_equipement     109365 non-null  object
 2   libde_equipement  109365 non-null  object
 3   libta_equipement  109365 non-null  object
 4   capinstot         84856 non-null   Int64 
 5   nofinessej        109365 non-null  object
 6   rs                109365 non-null  object
 7   rslongue          81940 non-null   object
 8   commune           109365 non-null  object
 9   departement       109365 non-null  object
 10  libdepartement    109365 non-null  object
 11  categetab         109365 non-null  object
 12  libcategetab      109365 non-null  object
 13  coordyet          109277 non-null  object
 14  coordxet          109277 non-null  object
dtypes: Int64(1), object(14)
memory usage: 12.6+ MB


In [3]:

# Création du transformateur Lambert 93 -> WGS84
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

# Conversion
df["longitude"], df["latitude"] = transformer.transform(
    df["coordxet"].values,df["coordyet"].values
)

# Aperçu
print(df[["nofinesset", "coordxet", "coordyet", "latitude", "longitude"]].head())

  nofinesset  coordxet   coordyet   latitude  longitude
0  920805066  643674.5  6864353.6  48.876739   2.232039
1  010010874  870430.3  6568991.9  46.199300   5.210436
2  010010874  870430.3  6568991.9  46.199300   5.210436
3  010010874  870430.3  6568991.9  46.199300   5.210436
4  020015392  706455.2  6906351.9  49.256794   3.088658


In [4]:
df.head()

,nofinesset,de_equipement,libde_equipement,libta_equipement,capinstot,nofinessej,rs,rslongue,commune,departement,libdepartement,categetab,libcategetab,coordyet,coordxet,longitude,latitude
0,920805066,897,Hébergement ouvert en foyer pour adultes handi...,Hébergement de Nuit Eclaté,12,920002268,APPARTEMENTS THERAPEUTIQUES,APPARTEMENTS THERAPEUTIQUES DE SURESNES ET PUT...,073,92,HAUTS DE SEINE,412,Appartement Thérapeutique,6864353.6,643674.5,2.232039,48.876739
1,010010874,507,Hébergement médico soc personnes en difficulté...,Hébergement de Nuit Eclaté,21,750045072,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,053,01,AIN,165,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,5.210436,46.199300
2,010010874,508,Accueil orientation soins accompagnement diff ...,Prestation en milieu ordinaire,15,750045072,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,053,01,AIN,165,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,5.210436,46.199300
3,010010874,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,4,750045072,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,053,01,AIN,165,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,5.210436,46.199300
4,020015392,507,Hébergement médico soc personnes en difficulté...,Hébergement de Nuit Eclaté,12,780020715,ACT CENTRE HENRI VINCENT,ACT CENTRE HENRI VINCENT - FONDATION DIACONESS...,810,02,AISNE,165,Appartement de Coordination Thérapeutique (A.C...,6906351.9,706455.2,3.088658,49.256794


In [5]:
df['libcategetab'].value_counts()

libcategetab
Etablissement d'hébergement pour personnes âgées dépendantes    25451
Service autonomie aide (SAA)                                    19731
Institut Médico-Educatif (I.M.E.)                                6326
Résidences autonomie                                             5572
Maison d'Enfants à Caractère Social                              4008
                                                                ...  
Etablissement de Soins Médicaux                                     1
Groupement de coopération sanitaire de moyens                       1
Service Médico-Psychologique Régional (S.M.P.R.)                    1
Service d'Aide aux Personnes Agées                                  1
Service de Travailleuses Familiales                                 1
Name: count, Length: 108, dtype: int64

In [6]:
# Regroupement des catégories de types d'établissements
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["hospital", "clinique", "soins", "dialyse", "ssr", "had", "cancer"]):
        return "Hopitaux cliniques"

    if any(x in cat for x in ["handicap", "ime", "itep", "mas", "fam", "esat", "sensoriel", "e.s.a.t.", "i.m.e.", "i.t.e.p.", "m.a.s.", "s.a.v.s.", "foyer de vie", "entreprise adaptée", "esat", "institut d'éducation motrice", "service mandataire judiciaire à la protection des majeurs", "c.a.m.s.p."]):
        return "Médico-social handicap"

    if any(x in cat for x in ["ehpad", "ehpa", "autonomie", "personnes âgées", "longue durée", "personnes agées"]):
        return "Personnes âgées"

    if any(x in cat for x in ["chrs", "casa", "c.a.d.a", "hébergement", "foyer", "maison relais"]):
        return "Social / Hébergement"

    if any(x in cat for x in ["pharmacie", "centre de santé", "maison de santé", "laboratoire", "c.m.p.", "cabinet", "infirmier", "kinésithérapeute", "dentiste", "opticien", "structure dispensatrice à domicile d'oxygène à usage médical", "maison médicale de garde (MMG)"]):
        return "Soins de ville"

    if any(x in cat for x in ['prévention', "vaccination", "dépistage", "csapa", "caarud" ]):
        return "Prévention / Santé publique"

    if any(x in cat for x in ["enfance", "camsp", "aemo", "aed", "pouponnière", "maison d'enfants", "educative", "protection maternelle et infantile", "protection infantile", "pmi", "p.m.i."]):
        return "Enfance / Protection"

    if any(x in cat for x in ["cpts", "mdph", "coordination", "groupement"]):
        return "Coordination / Administration"
    if any(x in cat for x in ["centre d'accueil", "accueil"]):
            return "Centres d'accueil"
    if any(x in cat for x in ["ecoles"]):
        return "Ecoles"
    return "Autres"

# Application
df["groupe"] = df["libcategetab"].apply(regrouper_categorie)

In [19]:
df['libde_equipement'].value_counts()

libde_equipement
Aide à Domicile                                          20519
Accueil pour Personnes Âgées                             14510
Accueil temporaire pour Personnes Âgées                   6475
Accueil au titre de la protection de l'enfance            5374
Tous projets éducatifs thérapeutiques et pédagogiques     5151
                                                         ...  
Placement Familial pour Enfants Handicapés                   2
Accueil collectif polyvalent regulier et occasionnel         1
Stationnement Pour Nomades                                   1
Tutelle d'État                                               1
Evaluat.réentraînem.orientat. scolaire cérébro-lésés         1
Name: count, Length: 123, dtype: int64

In [ ]:
# Regroupement des catégories d'activités
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["personnes agées"]):
        return "Personnes agées"

    if any(x in cat for x in ["enfance", "enfant", "éducatifs", "éducatif", "éducative", "enfants"]):
        return "Enfance"

    if any(x in cat for x in ["handicapés", "handicap", "handicapées", "adapté"]):
        return "Handicap"
    if any(x in cat for x in ["résidences sociales"]):
        return "Résidences sociales"
    if any(x in cat for x in ["hébergement médico soc"]):
        return "Hébergement médico soc"
    if any(x in cat for x in ["diff spécifiques"]):
        return "Difficultés spécifiques"
    if any(x in cat for x in ["à domicile"]):
        return "Aide et soins à domicile"
    if any(x in cat for x in ["amp dpn"]):
        return "AMP DPN"
    if any(x in cat for x in ["cancer"]):
        return "Cancer"
    if any(x in cat for x in ["gynécologie"]):
        return "Gynécologie"
    if any(x in cat for x in ["soins de longue durée"]):
        return "Soins de longue durée"
    if any(x in cat for x in ["insuffisance rénale"]):
        return "Insuffisance rénale"
    if any(x in cat for x in ["examen des caractéristiques génétiques"]):
        return "Examen des caractéristiques génétiques"
    if any(x in cat for x in ["urgence"]):
        return "Accueil et hébergement d'urgence"
    if any(x in cat for x in ["hébergement "]):
        return "Hébergement"
    if any(x in cat for x in ["accueil"]):
        return "Accueil"
    if any(x in cat for x in ["social"]):
        return "Social"
    if any(x in cat for x in ["réinsertion", "adaptation", "réhabilitation"]):
        return "Réinsertion et adaptation"
    if any(x in cat for x in ["psychologique"]):
        return "Aide psychologique"
    return "Autres"

# Application
df["groupe_equipement"] = df["libde_equipement"].apply(regrouper_categorie)

In [31]:
df['groupe_equipement'].value_counts()

groupe_equipement
Aide et soins à domicile            24653
Accueil                             21435
Handicap                            21110
Autres                              18635
Hébergement                          9094
Enfance                              5941
Accueil et hébergement d'urgence     3020
Social                               2172
Résidences sociales                  1294
Difficultés spécifiques              1212
Hébergement médico soc                656
Réinsertion et adaptation             123
Aide psychologique                     20
Name: count, dtype: int64

In [34]:
df.loc[df['groupe_equipement'] == "Autres", 'libde_equipement'].value_counts()


libde_equipement
Tous projets éducatifs thérapeutiques et pédagogiques           5151
Pôles d'activité et de soins adaptés                            2538
Section Cure Médicale (dont)                                    1680
Acc. dans l'acquisition de l'autonomie et la scolarisation      1621
Action Éducative en Milieu Ouvert                               1010
Accompagnement précoce de jeunes enfants                         864
Maisons Relais "Classique"                                       825
Activité soins d'accompagnement et de réhabilitation             599
Activité C.M.P.P.                                                488
Préparation à la vie professionnelle                             484
Plateforme d'accompagnement et de répit des aidants (PFR)        422
Centre de ressources territorial pour les personnes âgées        349
Activ.Club et Équipe de Prévention                               319
Information,conseil, expertise, coordination                     293
Tutelle curatelle

In [11]:
#nettoyage des doublons
df_clean = df.drop_duplicates()

In [12]:
#je crée un colonne code_insee à partir de la colonne code commune et de la colonne departement
df_clean.loc[:, "code_insee"] = (
    df_clean["departement"].astype(str).str.zfill(2)
    + df_clean["commune"].astype(str).str.zfill(3)
)

df_clean.head()

/tmp/ipykernel_9123/2999015001.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.loc[:, "code_insee"] = (


,nofinesset,de_equipement,libde_equipement,libta_equipement,capinstot,nofinessej,rs,rslongue,commune,departement,libdepartement,categetab,libcategetab,coordyet,coordxet,longitude,latitude,groupe,groupe_equipement,code_insee
0,920805066,897,Hébergement ouvert en foyer pour adultes handi...,Hébergement de Nuit Eclaté,12,920002268,APPARTEMENTS THERAPEUTIQUES,APPARTEMENTS THERAPEUTIQUES DE SURESNES ET PUT...,073,92,HAUTS DE SEINE,412,Appartement Thérapeutique,6864353.6,643674.5,2.232039,48.876739,Autres,Handicap,92073
1,010010874,507,Hébergement médico soc personnes en difficulté...,Hébergement de Nuit Eclaté,21,750045072,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,053,01,AIN,165,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,5.210436,46.199300,Coordination / Administration,Hébergement médico soc,01053
2,010010874,508,Accueil orientation soins accompagnement diff ...,Prestation en milieu ordinaire,15,750045072,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,053,01,AIN,165,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,5.210436,46.199300,Coordination / Administration,Difficultés spécifiques,01053
3,010010874,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,4,750045072,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,053,01,AIN,165,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,5.210436,46.199300,Coordination / Administration,Hébergement médico soc,01053
4,020015392,507,Hébergement médico soc personnes en difficulté...,Hébergement de Nuit Eclaté,12,780020715,ACT CENTRE HENRI VINCENT,ACT CENTRE HENRI VINCENT - FONDATION DIACONESS...,810,02,AISNE,165,Appartement de Coordination Thérapeutique (A.C...,6906351.9,706455.2,3.088658,49.256794,Coordination / Administration,Hébergement médico soc,02810


In [13]:
# latitudes longitudes en float
df_clean.loc[:,'latitude'] = df_clean['latitude'].astype(float)
df_clean.loc[:,'longitude'] = df_clean['longitude'].astype(float)

In [14]:
#je filtre les données sur les départements de la région Occitanie
departements_occitanie = ["09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82",  ]
df_occitanie = df_clean[df_clean["departement"].isin(departements_occitanie)]

In [15]:
df_occitanie.head()

,nofinesset,de_equipement,libde_equipement,libta_equipement,capinstot,nofinessej,rs,rslongue,commune,departement,libdepartement,categetab,libcategetab,coordyet,coordxet,longitude,latitude,groupe,groupe_equipement,code_insee
26,090003922,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,14,310026133,UCRM - ACT - ANTENNE PAMIERS,None,225,09,ARIEGE,165,Appartement de Coordination Thérapeutique (A.C...,6225284.3,587186.9,1.614832,43.117920,Coordination / Administration,Hébergement médico soc,09225
28,110003068,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,16,750015968,ACT GROUPE SOS SOLIDARITES,ACT SOS HABITAT ET SOINS CARCASSONNE,069,11,AUDE,165,Appartement de Coordination Thérapeutique (A.C...,6236151.6,648325.9,2.364397,43.222698,Coordination / Administration,Hébergement médico soc,11069
29,120007562,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,6,120783931,ACT VILLEFRANCHE DE ROUERGUE,APPARTEMENT DE COORDINATION THERAPEUTIQUE,300,12,AVEYRON,165,Appartement de Coordination Thérapeutique (A.C...,6361484.2,623238.6,2.037044,44.348176,Coordination / Administration,Hébergement médico soc,12300
94,300003399,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,18,750015968,ACT LOU CANTOU NIMES,APPARTEMENTS DE COORDINATION THERAPEUTIQUE LOU...,189,30,GARD,165,Appartement de Coordination Thérapeutique (A.C...,6305146.3,809570.4,4.362277,43.836821,Coordination / Administration,Hébergement médico soc,30189
95,300012259,507,Hébergement médico soc personnes en difficulté...,Hébergement de Nuit Eclaté,9,300786324,ACT AGFAS ALES,ACT AGFAS ALES,007,30,GARD,165,Appartement de Coordination Thérapeutique (A.C...,6336873.3,787150.3,4.089009,44.125456,Coordination / Administration,Hébergement médico soc,30007


In [16]:
df_occitanie.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9195 entries, 26 to 109255
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   nofinesset         9195 non-null   object 
 1   de_equipement      9195 non-null   object 
 2   libde_equipement   9195 non-null   object 
 3   libta_equipement   9195 non-null   object 
 4   capinstot          8440 non-null   Int64  
 5   nofinessej         9195 non-null   object 
 6   rs                 9195 non-null   object 
 7   rslongue           8169 non-null   object 
 8   commune            9195 non-null   object 
 9   departement        9195 non-null   object 
 10  libdepartement     9195 non-null   object 
 11  categetab          9195 non-null   object 
 12  libcategetab       9195 non-null   object 
 13  coordyet           9192 non-null   object 
 14  coordxet           9192 non-null   object 
 15  longitude          9192 non-null   float64
 16  latitude           9192 no

In [17]:
#j'exporte en csv dans processed la partie occitanie

df_occitanie.to_csv("../data/processed/equipements_occitanie.csv", index=False, sep=";", encoding="utf-8")
